# 🕸️ Agent Communication Patterns in LangGraph## Learning ObjectivesIn this notebook, you will learn:1. **Message Passing** - How agents can communicate by appending messages to a shared, growing conversation list2. **Shared State (Typed Fields)** - How agents can communicate through structured, typed state fields instead of free-form messages3. **Blackboard Pattern** - How multiple agents can read and write a shared workspace to iteratively refine a piece of work4. **Routing and Loops** - How to use conditional edges to loop a graph back on itself until a stopping condition is met## Prerequisites- Basic understanding of LangGraph (`StateGraph`, nodes, edges, `START`/`END`)- Familiarity with `TypedDict` state schemas and `Annotated` reducers (e.g. `add_messages`, `operator.add`)- An `OPENAI_API_KEY` set in a `.env` file at the project root

---## 🔧 Part 0: Environment SetupWe import the LangGraph and LangChain building blocks used throughout this notebook and initialize a single shared LLM instance. Every pattern below (message passing, shared state, blackboard) reuses this same `llm` object.

In [ ]:
# ============================================================================# ENVIRONMENT SETUP: Imports and LLM Initialization# ============================================================================# Standard libraryimport jsonimport operatorfrom typing import Literal# Third-partyfrom dotenv import load_dotenvfrom pydantic import BaseModel, Fieldfrom typing_extensions import Annotated, TypedDict# LangChain / LangGraphfrom langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessagefrom langchain_openai import ChatOpenAIfrom langgraph.graph import END, START, StateGraphfrom langgraph.graph.message import add_messagesload_dotenv()llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)print(f"✅ Environment ready. LLM initialized: {llm.model_name}")

---## 💬 Part 1: Message Passing PatternThe simplest way for agents to communicate in LangGraph is to share a single, append-only list of messages. Each agent reads everything written so far and appends its own message, so downstream agents automatically see upstream output.### Key Concepts:- **Shared message list**: A state field annotated with `add_messages` that accumulates messages instead of overwriting them- **Implicit context**: Each agent's prompt includes the full message history, so no explicit hand-off logic is needed

### 1.1 📋 `MessagePassingState`The state schema holds the growing `messages` list plus a `current_phase` field used to track which agent should act next.

In [ ]:
# ============================================================================# MESSAGE PASSING: State Schema# ============================================================================class MessagePassingState(TypedDict):    messages: Annotated[list[BaseMessage], add_messages]    current_phase: str

### 1.2 🏗️ `create_message_passing_pipeline`Three agents — a researcher, a fact-checker, and a summarizer — run in sequence. Each one reads the entire `messages` list built up so far and appends its own labeled contribution.

In [ ]:
# ============================================================================# MESSAGE PASSING: Building the Pipeline# ============================================================================def create_message_passing_pipeline():    """Agents communicate by appending messages that others can read."""    def researcher(state: MessagePassingState) -> dict:        """Researches the topic and posts findings as a message."""        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a researcher. Read the user's question, "                        "research it, and post your findings. Keep it to 2-3 sentences."                    )                ),                *state["messages"],            ]        )        return {            "messages": [                AIMessage(                    content=f"[RESEARCHER]: {response.content}", name="researcher"                )            ],            "current_phase": "fact_checker",        }    def fact_checker(state: MessagePassingState) -> dict:        """Reads the researcher's message and validates the claims."""        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a fact-checker. Read the researcher's findings "                        "in the conversation and validate or challenge them. "                        "Keep it to 2-3 sentences."                    )                ),                *state["messages"],            ]        )        return {            "messages": [                AIMessage(                    content=f"[FACT-CHECKER]: {response.content}", name="fact_checker"                )            ],            "current_phase": "summarizer",        }    def summarizer(state: MessagePassingState) -> dict:        """Reads all previous messages and creates a final summary."""        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a summarizer. Read the researcher's findings and "                        "the fact-checker's review. Produce a final, accurate summary. "                        "Keep it to 2-3 sentences."                    )                ),                *state["messages"],            ]        )        return {            "messages": [                AIMessage(content=f"[SUMMARY]: {response.content}", name="summarizer")            ],            "current_phase": "done",        }    graph = StateGraph(MessagePassingState)    graph.add_node("researcher", researcher)    graph.add_node("fact_checker", fact_checker)    graph.add_node("summarizer", summarizer)    graph.add_edge(START, "researcher")    graph.add_edge("researcher", "fact_checker")    graph.add_edge("fact_checker", "summarizer")    graph.add_edge("summarizer", END)    return graph.compile()print("✅ Message passing pipeline defined: researcher -> fact_checker -> summarizer")

### 1.3 ▶️ `demo_message_passing`Runs the pipeline on a sample question and prints each agent's labeled message in order, showing how the conversation accumulates.

In [ ]:
# ============================================================================# MESSAGE PASSING: Demo# ============================================================================def demo_message_passing():    """Demo message passing between agents."""    agent = create_message_passing_pipeline()    print("Message Passing Demo:\n")    result = agent.invoke(        {            "messages": [                HumanMessage(content="What are the main benefits of renewable energy?")            ],            "current_phase": "researcher",        }    )    for msg in result["messages"]:        if isinstance(msg, AIMessage):            print(f"{msg.content}\n")

---## 🗂️ Part 2: Shared State (Typed Fields) PatternInstead of a free-form message log, agents can communicate through **structured, typed state fields**. Each agent writes to its own field(s), and downstream agents read those fields directly — no need to parse natural-language history.### Key Concepts:- **Typed fields**: Each piece of information (raw data, analysis, recommendations) has its own named, typed slot in state- **Reducers**: `Annotated[list[dict], operator.add]` lets multiple writes accumulate instead of overwriting

### 2.1 📋 `SharedFieldsState`Defines one field per stage of the pipeline: the query, accumulated `raw_data`, the analyst's `analysis` and `confidence_score`, and the advisor's `recommendations`.

In [ ]:
# ============================================================================# SHARED STATE: State Schema# ============================================================================class SharedFieldsState(TypedDict):    query: str    # Each agent writes to its own field — others can read it    raw_data: Annotated[list[dict], operator.add]    analysis: str    recommendations: list[str]    confidence_score: float

### 2.2 🏗️ `create_shared_fields_pipeline`A data collector, an analyst, and an advisor run in sequence. Each one reads only the specific fields it needs from state, and writes back its own fields — no shared conversational context required.

In [ ]:
# ============================================================================# SHARED STATE: Building the Pipeline# ============================================================================def create_shared_fields_pipeline():    """Agents communicate through typed state fields, not messages."""    def data_collector(state: SharedFieldsState) -> dict:        """Collects data and writes to the raw_data field."""        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a data collector. Given the query, produce 3 data points "                        "as a JSON array of objects with 'source' and 'finding' keys. "                        "Return ONLY the JSON array, no markdown."                    )                ),                HumanMessage(content=state["query"]),            ]        )        try:            data = json.loads(response.content)        except json.JSONDecodeError:            data = [{"source": "llm", "finding": response.content}]        return {"raw_data": data}    def analyst(state: SharedFieldsState) -> dict:        """Reads raw_data field, writes analysis and confidence."""        data_summary = json.dumps(state["raw_data"], indent=2)        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a data analyst. Analyze the collected data and provide: "                        "1) A brief analysis (2-3 sentences), and "                        "2) A confidence score from 0.0 to 1.0. "                        "Format: ANALYSIS: <text>\nCONFIDENCE: <number>"                    )                ),                HumanMessage(                    content=f"Query: {state['query']}\n\nData:\n{data_summary}"                ),            ]        )        content = response.content        analysis = content        confidence = 0.7  # default        if "CONFIDENCE:" in content:            parts = content.split("CONFIDENCE:")            analysis = parts[0].replace("ANALYSIS:", "").strip()            try:                confidence = float(parts[1].strip())            except ValueError:                confidence = 0.7        return {"analysis": analysis, "confidence_score": confidence}    def advisor(state: SharedFieldsState) -> dict:        """Reads analysis + confidence, writes recommendations."""        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a strategic advisor. Based on the analysis and "                        "confidence score, provide 3 actionable recommendations. "                        "Return them as a JSON array of strings. "                        "Return ONLY the JSON array, no markdown."                    )                ),                HumanMessage(                    content=(                        f"Query: {state['query']}\n"                        f"Analysis: {state['analysis']}\n"                        f"Confidence: {state['confidence_score']}"                    )                ),            ]        )        try:            recs = json.loads(response.content)        except json.JSONDecodeError:            recs = [response.content]        return {"recommendations": recs}    graph = StateGraph(SharedFieldsState)    graph.add_node("data_collector", data_collector)    graph.add_node("analyst", analyst)    graph.add_node("advisor", advisor)    graph.add_edge(START, "data_collector")    graph.add_edge("data_collector", "analyst")    graph.add_edge("analyst", "advisor")    graph.add_edge("advisor", END)    return graph.compile()print("✅ Shared state pipeline defined: data_collector -> analyst -> advisor")

### 2.3 ▶️ `demo_shared_state`Runs the pipeline on a sample business question and prints the collected data, the analysis, the confidence score, and the final recommendations.

In [ ]:
# ============================================================================# SHARED STATE: Demo# ============================================================================def demo_shared_state():    """Demo shared state fields between agents."""    agent = create_shared_fields_pipeline()    print("Shared State Demo:\n")    result = agent.invoke(        {            "query": "Should a small business invest in AI automation in 2026?",            "raw_data": [],            "analysis": "",            "recommendations": [],            "confidence_score": 0.0,        }    )    print(f"Data collected: {len(result['raw_data'])} points")    for d in result["raw_data"]:        print(f"  - [{d.get('source', 'N/A')}] {d.get('finding', 'N/A')[:80]}...")    print(f"\nAnalysis: {result['analysis'][:200]}...")    print(f"Confidence: {result['confidence_score']}")    print(f"\nRecommendations:")    for i, rec in enumerate(result["recommendations"], 1):        print(f"  {i}. {rec}")

---## 🖼️ Part 3: Blackboard PatternIn the blackboard pattern, multiple agents read from and write to a **shared workspace** (the "blackboard") and iterate until a stopping condition is met. Here a drafter and a critic repeatedly revise a paragraph until it's approved or a maximum number of iterations is reached.### Key Insight:> The blackboard's `drafts` and `critiques` lists act as a shared history that both agents can see. A conditional edge routes the graph back to the drafter until the critic approves — this is what makes the pattern an iterative loop rather than a fixed pipeline.

### 3.1 📋 `BlackboardState`Alongside the message log, the blackboard holds the `topic`, the running list of `drafts` and `critiques`, an `iteration` counter, and an `is_approved` flag used to control the loop.

In [ ]:
# ============================================================================# BLACKBOARD: State Schema# ============================================================================class BlackboardState(TypedDict):    messages: Annotated[list[BaseMessage], add_messages]    # Blackboard fields — the shared workspace    topic: str    drafts: Annotated[list[str], operator.add]    critiques: Annotated[list[str], operator.add]    iteration: int    is_approved: bool

### 3.2 🏗️ `create_blackboard_system`The `drafter` reads the latest draft and critique from the blackboard and writes an improved draft. The `critic` reviews the latest draft with a structured-output LLM call and either approves it or leaves feedback. A conditional edge, `route_after_critic`, loops back to the drafter until approval (or a forced approval after 3 iterations, to guarantee the loop terminates).

In [ ]:
# ============================================================================# BLACKBOARD: Building the System# ============================================================================def create_blackboard_system():    """    Blackboard pattern: multiple agents read/write a shared workspace.    A drafter writes, a critic reviews, and they iterate until approved.    """    class ApprovalDecision(BaseModel):        approved: bool = Field(description="Whether the draft is good enough")        feedback: str = Field(description="Specific feedback if not approved")    critic_llm = llm.with_structured_output(ApprovalDecision)    def drafter(state: BlackboardState) -> dict:        """Reads critiques from blackboard, writes improved draft."""        context_parts = [f"Topic: {state['topic']}"]        if state["drafts"]:            context_parts.append(f"Previous draft: {state['drafts'][-1]}")        if state["critiques"]:            context_parts.append(f"Feedback to address: {state['critiques'][-1]}")        context = "\n".join(context_parts)        response = llm.invoke(            [                SystemMessage(                    content=(                        "You are a skilled writer. Write or revise a short paragraph "                        "(3-4 sentences) based on the topic and any feedback provided. "                        "If there's feedback, directly address it in your revision."                    )                ),                HumanMessage(content=context),            ]        )        return {            "drafts": [response.content],            "messages": [                AIMessage(                    content=f"[DRAFTER iteration {state['iteration'] + 1}]: {response.content}",                    name="drafter",                )            ],            "iteration": state["iteration"] + 1,        }    def critic(state: BlackboardState) -> dict:        """Reads latest draft from blackboard, writes critique or approves."""        latest_draft = state["drafts"][-1] if state["drafts"] else "No draft yet"        decision = critic_llm.invoke(            [                SystemMessage(                    content=(                        "You are a strict editor. Review the draft for clarity, accuracy, "                        "and engagement. Approve ONLY if it's genuinely good. "                        "If iteration is 3 or more, be more lenient."                    )                ),                HumanMessage(                    content=(                        f"Topic: {state['topic']}\n"                        f"Iteration: {state['iteration']}\n"                        f"Draft: {latest_draft}"                    )                ),            ]        )        # Force approval after 3 iterations to prevent infinite loops        approved = decision.approved or state["iteration"] >= 3        result = {            "is_approved": approved,            "messages": [                AIMessage(                    content=f"[CRITIC]: {'APPROVED' if approved else 'REVISION NEEDED'} - {decision.feedback}",                    name="critic",                )            ],        }        if not approved:            result["critiques"] = [decision.feedback]        return result    def route_after_critic(state: BlackboardState) -> Literal["drafter", "end"]:        """Loop back to drafter if not approved."""        if state["is_approved"]:            return "end"        return "drafter"    graph = StateGraph(BlackboardState)    graph.add_node("drafter", drafter)    graph.add_node("critic", critic)    graph.add_edge(START, "drafter")    graph.add_edge("drafter", "critic")    graph.add_conditional_edges(        "critic", route_after_critic, {"drafter": "drafter", "end": END}    )    return graph.compile()print("✅ Blackboard system defined: drafter <-> critic (loops until approved)")

### 3.3 ▶️ `demo_blackboard`Runs the drafter/critic loop on a sample topic and prints every iteration's message, the total number of iterations taken, and the final approved draft.

In [ ]:
# ============================================================================# BLACKBOARD: Demo# ============================================================================def demo_blackboard():    """Demo blackboard iterative refinement."""    agent = create_blackboard_system()    print("Blackboard Pattern Demo:\n")    result = agent.invoke(        {            "messages": [],            "topic": "Why LangGraph is great for building multi-agent systems",            "drafts": [],            "critiques": [],            "iteration": 0,            "is_approved": False,        }    )    print(f"Total iterations: {result['iteration']}")    print(f"Approved: {result['is_approved']}")    print("\nConversation:")    for msg in result["messages"]:        if isinstance(msg, AIMessage):            print(f"\n{msg.content}")    print(f"\nFinal draft:\n{result['drafts'][-1]}")

---## ▶️ Running the DemosThe original script's `__main__` guard is kept as-is below — Jupyter sets `__name__` to `"__main__"`, so this cell runs whichever demo is uncommented. Only `demo_blackboard()` is active by default; uncomment the others to try the message-passing or shared-state patterns instead.

In [ ]:
# ============================================================================# RUN: Execute a Demo# ============================================================================if __name__ == "__main__":    # demo_shared_state()    # print("\n" + "=" * 50 + "\n")    # demo_message_passing()    # print("\n" + "=" * 50 + "\n")    demo_blackboard()

---## 📝 SummaryIn this notebook, we learned three ways for agents in a LangGraph graph to communicate with each other:### 1. Message Passing- Agents share a single `messages` list (`Annotated[list[BaseMessage], add_messages]`)- Each agent reads the full history and appends its own labeled message- Simple and transparent, but the LLM must parse conversational context each time### 2. Shared State (Typed Fields)- Agents communicate through structured, typed state fields instead of free text- Reducers like `operator.add` let multiple agents accumulate into the same field- More structured and easier to validate than raw messages, at the cost of a more rigid schema### 3. Blackboard Pattern- Multiple agents read/write a shared workspace (`drafts`, `critiques`) and iterate- A conditional edge (`route_after_critic`) loops the graph until an approval condition is met- A hard iteration cap (`iteration >= 3`) guarantees the loop terminates### Next Steps- Explore how these communication patterns combine with **supervisor** and **swarm** multi-agent orchestration patterns- Try swapping the blackboard's iteration cap for a token-budget or timeout-based stopping condition